# gap 건너기 하드코딩 데모 (10대 일렬)

학습 없이, 정해진 순서대로 움직이게 해서 10대가 틈을 건너는 장면을 만듭니다.

**동작 방식**

1. 앞쪽 3대를 **다리(널빤지)**로 씁니다. 틈 위에 겹쳐 놓고 `lock_root_motion`으로 고정해, 물리적으로 떨어지지 않습니다.
2. 나머지 7대가 순서대로 그 위를 건넙니다.

**두 가지 모드**

| MODE | 내용 | 결과 |
|---|---|---|
| `scripted` (기본) | 건너는 로봇의 **위치까지 직접 지시**합니다. 몸은 자벌레 동작을 그대로 하지만, 앞으로 나아가는 것은 대본대로입니다. | 데모가 확실히 완성됩니다 |
| `physics` | 다리만 고정하고, 나머지는 **스스로 걷습니다**. | 실제 걸음 성능에 달려 있어, 느리거나 실패할 수 있습니다 |

`FLUSH_BRIDGE = True`면 다리 윗면을 플랫폼 높이에 맞춰서 **올라타는 턱을 없앱니다**. 지금 풀고 있는 "올라타기" 문제를 피해 가려고 넣은 설정이에요. `False`로 두면 다리가 플랫폼 위에 얹혀 5 mm 턱이 생깁니다.

**주의:** 여기서는 MuJoCo를 돌릴 수 없어 실제로 실행해 보지 못했습니다. 문법과 좌표 계산만 확인했습니다. 다리가 건너편에 못 닿으면 `BRIDGE_COUNT`를 늘리라는 메시지가 나옵니다.

`BARISimulation` 폴더에 두고 **`barisimulation` 커널**로 위에서부터 실행하세요.

In [1]:
# 1) 준비 — 10대가 일렬(10*1)로 서서 gap을 건너는 장면을 하드코딩한다.
import math, time, gc
import numpy as np
import mujoco

from bari_sim.robot.actions import GripAction, LiftAction, MotionAction, RobotAction
from bari_sim.robot.specification import DEFAULT_ROBOT
from bari_sim.simulation import SceneRequest, Simulation
from bari_sim.tasks import parse_robot_grid, task_definition

# ---------------- 설정 ----------------
DIFFICULTY   = 3        # 1:0.10 m, 2:0.15, 3:0.20, 4:0.25, 5:0.30
ROBOTS       = 10       # 일렬로 몇 대
BRIDGE_COUNT = 3        # 다리로 쓸 로봇 수 (앞에서부터)
MODE         = "scripted"   # "scripted": 경로까지 지시(데모 보장) / "physics": 다리만 고정, 나머지는 스스로 걸음
WALK_SPEED   = 0.03     # scripted 모드에서 건너는 속도 (m/s)
QUEUE_GAP_S  = 2.5      # 뒤 로봇이 출발하기까지 기다리는 간격 (s)
DURATION_S   = 120.0
FLUSH_BRIDGE = True     # True면 다리 윗면을 플랫폼 높이에 맞춰 턱을 없앤다

task = task_definition("gap", DIFFICULTY)
GAP = task.value
L = DEFAULT_ROBOT.length_m          # 0.15
H = DEFAULT_ROBOT.height_m          # 0.005
REAR = DEFAULT_ROBOT.rear_length_m  # 0.06
EDGE_NEAR, EDGE_FAR = -GAP / 2.0, GAP / 2.0
print(f"틈 {GAP * 100:.0f} cm | 로봇 {ROBOTS}대 (다리 {BRIDGE_COUNT}대 + 건너는 {ROBOTS - BRIDGE_COUNT}대) | 모드 {MODE}")

sim = Simulation(SceneRequest(grid=parse_robot_grid(f"{ROBOTS}*1"), environment="gap", task=task))

def qpos_adr(rid):
    return int(sim.model.jnt_qposadr[sim.model.joint(f"robot_{rid}_root").id])

def dof_adr(rid):
    return int(sim.model.jnt_dofadr[sim.model.joint(f"robot_{rid}_root").id])

def place(rid, x, y=0.0, z=None, pitch_rad=0.0):
    """로봇의 몸통(뒤 조각 중심)을 (x, y, z)에 놓는다. z를 생략하면 바닥에 붙인다."""
    if z is None:
        z = H / 2.0
    a = qpos_adr(rid)
    sim.data.qpos[a:a + 3] = (x, y, z)
    sim.data.qpos[a + 3:a + 7] = (math.cos(pitch_rad / 2), 0.0, math.sin(pitch_rad / 2), 0.0)
    d = dof_adr(rid)
    sim.data.qvel[d:d + 6] = 0.0

def body_x(rid):
    return float(sim.data.xpos[sim.model.body(f"robot_{rid}_rear_body").id][0])

# 로봇 한 대가 차지하는 x 범위: [x - REAR/2, x - REAR/2 + L]
def x_for_rear_end(rear_end):
    return rear_end + REAR / 2.0

# ---------------- 다리 배치 계산 ----------------
# 틈을 덮도록 널빤지를 겹쳐 놓는다. 마지막 널은 건너편 플랫폼에 걸친다.
overlap = max(0.0, (BRIDGE_COUNT * L - (GAP + 2 * 0.04)) / max(BRIDGE_COUNT - 1, 1))
step_x = L - overlap
bridge_z = H / 2.0 - (H if FLUSH_BRIDGE else 0.0)   # 플러시면 윗면이 플랫폼 높이와 같아진다
BRIDGE_IDS = list(range(BRIDGE_COUNT))
bridge_pose = {}
rear_end = EDGE_NEAR - 0.04           # 첫 널은 이쪽 플랫폼에 4 cm 걸친다
for i, rid in enumerate(BRIDGE_IDS):
    bridge_pose[rid] = x_for_rear_end(rear_end + i * step_x)
span_end = rear_end + (BRIDGE_COUNT - 1) * step_x + L
print(f"다리: 널 {BRIDGE_COUNT}장, 겹침 {overlap * 100:.1f} cm, "
      f"덮는 구간 {rear_end * 100:+.1f} ~ {span_end * 100:+.1f} cm (건너편 가장자리 {EDGE_FAR * 100:+.1f} cm)")
assert span_end > EDGE_FAR + 0.03, "다리가 건너편에 못 닿습니다. BRIDGE_COUNT를 늘리세요."

WALKER_IDS = list(range(BRIDGE_COUNT, ROBOTS))
GOAL_X = EDGE_FAR + L / 2.0 + 0.05     # 건너편에서 멈출 위치 (성공 판정선보다 안쪽)


틈 20 cm | 로봇 10대 (다리 3대 + 건너는 7대) | 모드 scripted
다리: 널 3장, 겹침 8.5 cm, 덮는 구간 -14.0 ~ +14.0 cm (건너편 가장자리 +10.0 cm)


## 배치

In [2]:
# 2) 동작 정의 + 초기 배치
def act(motion=None, lift=None, grip=None):
    kwargs = {k: v for k, v in (("motion", motion), ("lift", lift), ("grip", grip)) if v is not None}
    return RobotAction(**kwargs)

MOTION_STOP = getattr(MotionAction, "STOP", None)
LIFT_STOP = getattr(LiftAction, "STOP", None)
GRIP_STOP = getattr(GripAction, "STOP", None)
HOLD = act(MOTION_STOP, LIFT_STOP, GRIP_STOP)
GAIT = [
    act(MotionAction.CURL_BODY, LiftAction.LIFT_FRONT, GRIP_STOP),
    act(MotionAction.CURL_BODY, LIFT_STOP, GRIP_STOP),
    act(MotionAction.FLATTEN_BODY, LiftAction.UNLIFT_FRONT, GRIP_STOP),
    act(MotionAction.FLATTEN_BODY, LIFT_STOP, GRIP_STOP),
]

sim.reset()

# 다리 로봇을 제자리에 놓는다
for rid, x in bridge_pose.items():
    place(rid, x, z=bridge_z)

# 건너는 로봇은 이쪽 플랫폼에 줄을 세운다 (앞 로봇부터 순서대로)
queue_x0 = EDGE_NEAR - 0.10
for k, rid in enumerate(WALKER_IDS):
    place(rid, queue_x0 - k * (L + 0.03))
mujoco.mj_forward(sim.model, sim.data)

start_x = {rid: body_x(rid) for rid in WALKER_IDS}
print("다리 로봇 x:", {rid: round(bridge_pose[rid], 3) for rid in BRIDGE_IDS})
print("건너는 로봇 x:", {rid: round(start_x[rid], 3) for rid in WALKER_IDS})


다리 로봇 x: {0: -0.11, 1: -0.045, 2: 0.02}
건너는 로봇 x: {3: -0.2, 4: -0.38, 5: -0.56, 6: -0.74, 7: -0.92, 8: -1.1, 9: -1.28}


## 실행

In [3]:
# 3) 실행 — 다리는 고정, 나머지는 순서대로 건넌다.
def run(record=None):
    """record: frame_callback (뷰어용) 또는 None"""
    crossed = set()
    step = 0
    log_at = 0.0
    while sim.time_s + 1e-9 < DURATION_S:
        t = sim.time_s
        actions = {}
        locked = list(BRIDGE_IDS)          # 다리는 언제나 고정
        for rid in BRIDGE_IDS:
            actions[rid] = HOLD

        for k, rid in enumerate(WALKER_IDS):
            released = t >= k * QUEUE_GAP_S          # 순서대로 출발
            arrived = body_x(rid) >= GOAL_X
            if arrived:
                crossed.add(rid)
            if MODE == "scripted":
                actions[rid] = HOLD if (not released or arrived) else GAIT[step % len(GAIT)]
                locked.append(rid)                   # 경로를 직접 지시하므로 물리 이동은 막는다
                if released and not arrived:
                    a = qpos_adr(rid)
                    x = float(sim.data.qpos[a]) + WALK_SPEED * sim.robot.control_interval_s
                    place(rid, min(x, GOAL_X), z=H / 2.0)
            else:                                     # physics 모드: 스스로 걷는다
                actions[rid] = GAIT[step % len(GAIT)] if released and not arrived else HOLD
        if MODE == "scripted":
            mujoco.mj_forward(sim.model, sim.data)

        sim.step(actions, lock_root_motion=locked,
                 frame_callback=record, render_hz=60.0 if record else 1.0,
                 realtime=bool(record))
        step += 1
        if t >= log_at:
            log_at += 10.0
            print(f"  t={t:5.1f}s  건넌 로봇 {len(crossed)}/{len(WALKER_IDS)}  "
                  f"선두 x={max(body_x(r) for r in WALKER_IDS) * 100:+6.1f} cm", flush=True)
    return crossed

t0 = time.perf_counter()
crossed = run()
print(f"\n결과: {len(crossed)}/{len(WALKER_IDS)}대가 건넜습니다. ({time.perf_counter() - t0:.0f}s 걸림)")
result = sim.evaluator.result() if sim.evaluator is not None else None
if result is not None:
    print("과제 판정:", {k: result.metrics.get(k) for k in ("score", "succeeded_robot_count")})
print("로봇별 최종 x(cm):", {rid: round(body_x(rid) * 100, 1) for rid in range(ROBOTS)})


  t=  0.0s  건넌 로봇 0/7  선두 x= -18.5 cm
  t= 10.0s  건넌 로봇 0/7  선두 x= +11.5 cm
  t= 20.5s  건넌 로봇 1/7  선두 x= +22.5 cm
  t= 30.5s  건넌 로봇 2/7  선두 x= +22.5 cm
  t= 40.0s  건넌 로봇 3/7  선두 x= +22.5 cm
  t= 50.0s  건넌 로봇 5/7  선두 x= +22.5 cm
  t= 60.0s  건넌 로봇 6/7  선두 x= +22.5 cm
  t= 70.0s  건넌 로봇 7/7  선두 x= +22.5 cm
  t= 80.5s  건넌 로봇 7/7  선두 x= +22.5 cm
  t= 90.5s  건넌 로봇 7/7  선두 x= +22.5 cm
  t=100.5s  건넌 로봇 7/7  선두 x= +22.5 cm
  t=110.5s  건넌 로봇 7/7  선두 x= +22.5 cm

결과: 7/7대가 건넜습니다. (166s 걸림)
과제 판정: {'score': -1.177, 'succeeded_robot_count': None}
로봇별 최종 x(cm): {0: -11.0, 1: -4.5, 2: 2.0, 3: 22.5, 4: 22.5, 5: 22.5, 6: 22.5, 7: 22.5, 8: 22.5, 9: 22.5}


## 뷰어 (선택)

In [5]:
# 4) (선택) 뷰어로 다시 보기 — 창을 닫으면 끝난다.
import mujoco.viewer

sim.reset()
for rid, x in bridge_pose.items():
    place(rid, x, z=bridge_z)
for k, rid in enumerate(WALKER_IDS):
    place(rid, queue_x0 - k * (L + 0.03))
mujoco.mj_forward(sim.model, sim.data)

with mujoco.viewer.launch_passive(sim.model, sim.data) as viewer:
    run(record=lambda s: viewer.sync() if viewer.is_running() else False)


  t=  0.0s  건넌 로봇 0/7  선두 x= -18.5 cm
  t= 10.0s  건넌 로봇 0/7  선두 x= +11.5 cm
  t= 20.5s  건넌 로봇 1/7  선두 x= +22.5 cm
  t= 30.5s  건넌 로봇 2/7  선두 x= +22.5 cm
  t= 40.0s  건넌 로봇 3/7  선두 x= +22.5 cm
  t= 50.0s  건넌 로봇 5/7  선두 x= +22.5 cm


KeyboardInterrupt: 